# Bronze Layer — osm_ingest (OpenStreetMap food places in Ho Chi Minh City, weekly)

Added 2026-10-31 (proposal "nguồn dữ liệu khác": OpenStreetMap). Free, no API key, licence **ODbL**: anything built on it must show **"© OpenStreetMap contributors"**.

**Flow:**
1. Skip when the last successful ingest is younger than `REFRESH_DAYS` (7): OSM changes slowly and the public Overpass servers ask for light use.
2. One **Overpass API** query: every node / way / relation tagged `amenity` in `AMENITIES` inside the Ho Chi Minh City boundary (`ISO3166-2 = VN-SG`, the post-2025 city), with its tags and centre point (`out center tags`). Mirrors in `OVERPASS_URLS` are tried in order.
3. **Append** the snapshot to `${bronze}.osm_pois_raw` (one row per OSM element per ingest; tags kept as a map, nothing dropped).

**Never fails the job:** a timeout, an HTTP error or a suspiciously small answer (< `MIN_ELEMENTS`) is printed and nothing is written; the next daily run tries again. `stg/osm_restaurants` then keeps using the last good snapshot.

In [ ]:
# --- Config ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

TARGET = f"{BRONZE}.osm_pois_raw"
REFRESH_DAYS = 7       # one Overpass query per week
MIN_ELEMENTS = 500     # fewer = broken answer (HCMC has thousands of food places in OSM); nothing is written
AREA_FILTER = '["ISO3166-2"="VN-SG"]["boundary"="administrative"]'   # Thành phố Hồ Chí Minh
AMENITIES = ["restaurant", "cafe", "fast_food", "food_court", "bar", "pub", "ice_cream", "biergarten"]
OVERPASS_URLS = ["https://overpass-api.de/api/interpreter",
                 "https://overpass.private.coffee/api/interpreter"]
USER_AGENT = "project1-youtube-restaurants/1.0 (Databricks weekly job; personal learning project)"

QUERY = f"""[out:json][timeout:300];
area{AREA_FILTER}->.city;
nwr["amenity"~"^({'|'.join(AMENITIES)})$"](area.city);
out center tags;"""
print(f"Bronze target: {TARGET}")
print(QUERY)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET} (
        osm_type     STRING,
        osm_id       BIGINT,
        latitude     DOUBLE,
        longitude    DOUBLE,
        tags         MAP<STRING, STRING>,
        osm_base     TIMESTAMP,
        ingest_ts    TIMESTAMP,
        ingest_date  DATE
    )
    USING DELTA
    COMMENT 'OpenStreetMap food places (amenity restaurant / cafe / fast_food / ...) in Ho Chi Minh City, one snapshot per weekly Overpass query (append). Data © OpenStreetMap contributors, ODbL.'
""")
last = spark.sql(f"SELECT MAX(ingest_ts) AS ts, current_timestamp() AS now FROM {TARGET}").collect()[0]
due = last["ts"] is None or (last["now"] - last["ts"]).total_seconds() >= REFRESH_DAYS * 86400 - 3600
print(f"Last ingest: {last['ts']} · due: {due}")

In [ ]:
# --- Query Overpass (never raises) ---
import requests
from datetime import datetime

elements, osm_base, error = [], None, None
if due:
    for url in OVERPASS_URLS:
        try:
            resp = requests.post(url, data={"data": QUERY}, headers={"User-Agent": USER_AGENT}, timeout=360)
            if resp.status_code != 200:
                error = f"{url}: HTTP {resp.status_code} {resp.text[:200]}"
                print("⚠️", error)
                continue
            payload = resp.json()
            elements = payload.get("elements", [])
            base = (payload.get("osm3s") or {}).get("timestamp_osm_base")
            osm_base = datetime.fromisoformat(base.replace("Z", "+00:00")).replace(tzinfo=None) if base else None
            if len(elements) >= MIN_ELEMENTS:
                error = None
                print(f"✅ {url}: {len(elements)} elements (OSM data as of {osm_base})")
                break
            error = f"{url}: only {len(elements)} elements (< {MIN_ELEMENTS}), answer ignored"
            print("⚠️", error)
            elements = []
        except Exception as e:   # timeout, connection refused, bad JSON
            error = f"{url}: {type(e).__name__}: {str(e)[:200]}"
            print("⚠️", error)
else:
    print(f"⏭️ Skipped: last ingest is younger than {REFRESH_DAYS} days")

In [ ]:
# --- Append the snapshot ---
from pyspark.sql.functions import current_timestamp, to_date, from_utc_timestamp, col

rows = []
for e in elements:
    center = e.get("center") or {}
    lat, lon = e.get("lat", center.get("lat")), e.get("lon", center.get("lon"))
    if lat is None or lon is None:
        continue
    rows.append((e.get("type"), int(e["id"]), float(lat), float(lon), dict(e.get("tags") or {}), osm_base))
if rows:
    df = (spark.createDataFrame(rows, "osm_type STRING, osm_id BIGINT, latitude DOUBLE, longitude DOUBLE, "
                                      "tags MAP<STRING, STRING>, osm_base TIMESTAMP")
          .withColumn("ingest_ts", current_timestamp())
          .withColumn("ingest_date", to_date(from_utc_timestamp(col("ingest_ts"), "Asia/Ho_Chi_Minh"))))
    df.write.format("delta").mode("append").saveAsTable(TARGET)
    print(f"✅ Appended {len(rows)} OSM elements to {TARGET}")
elif due:
    print(f"⚠️ Nothing written ({error}); stg/osm_restaurants keeps the last snapshot, the next run retries")
dbutils.jobs.taskValues.set("osm_elements", len(rows))
display(spark.sql(f"""
    SELECT ingest_date, tags['amenity'] AS amenity, COUNT(*) AS n
    FROM {TARGET} WHERE ingest_ts = (SELECT MAX(ingest_ts) FROM {TARGET})
    GROUP BY 1, 2 ORDER BY n DESC
"""))
print("\n--- osm_ingest complete ---")